## Example 1: Tabula Sapiens

In [1]:
# Run this once in R
# Install if needed
# Install if needed
if (!requireNamespace("sceasy", quietly = TRUE)) {
  devtools::install_github("cellgeni/sceasy")
}

library(sceasy)
library(reticulate)


Loading required package: reticulate



In [3]:
library(Matrix)

# Read your RDS files
sparse_matrix <- readRDS("/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_ref/scGX_rds_files/TabulaSapiensMatrixSampled.rds")
metadata <- readRDS("/mnt/lscratch/users/adhal/SingleCellUtils/data/scHIIT_ref/scGX_rds_files/TabulaSapiensMetaSampled.rds")  # or read.csv if it's CSV

In [ ]:
metadata$Annot

Cell,Annot
<chr>,<chr>
ACACCAAGTCGCTTGG_TSP14_Bladder_NA_10X_1_1,T.Cell
CACAGGCAGCCGCTTG_TSP14_Bladder_NA_10X_1_1,T.Cell
CACTGTCGTCTGTCCT_TSP14_Bladder_NA_10X_1_1,T.Cell
CATGCGGAGACCAGAC_TSP14_Bladder_NA_10X_1_1,T.Cell
CCTATCGGTATTCCGA_TSP14_Bladder_NA_10X_1_1,T.Cell
CTTACCGCAGCGTTGC_TSP14_Bladder_NA_10X_1_1,T.Cell
GTGGAAGTCTGCCCTA_TSP14_Bladder_NA_10X_1_1,T.Cell
GTTCATTTCGCAGTGC_TSP14_Bladder_NA_10X_1_1,T.Cell
GCTGGGTAGATTGATG_TSP14_Bladder_NA_10X_1_2,T.Cell


In [ ]:
# Export matrix in Matrix Market format
writeMM(sparse_matrix, "matrix.mtx")

# Export gene names (row names)
writeLines(rownames(sparse_matrix), "genes.txt")

# Export cell barcodes (column names)
writeLines(colnames(sparse_matrix), "barcodes.txt")

# Export metadata as CSV
# Make sure the row names match the cell barcodes
write.csv(metadata, "metadata.csv", row.names = TRUE)

## Example 2

In [ ]:
library(Seurat)
library(Matrix)

# Load data
data <- readRDS("/mnt/lscratch/users/adhal/data/scrna_target_idf_v2/scrna_target_idf_v2/data/Immune_data/CD4_naive_to_Th17.rds")

# Get counts matrix
counts <- LayerData(data, assay = "RNA", layer = "counts")
# Or for older Seurat: counts <- GetAssayData(data, slot = "counts")

# Export directory
export_dir <- "/mnt/lscratch/users/adhal/scrna_target_idf_v3/data/from_martin/"

# 1. Save counts as Matrix Market format (efficient for sparse matrices)
writeMM(counts, file.path(export_dir, "counts.mtx"))

# 2. Save gene names
write.csv(data.frame(gene = rownames(counts)), 
          file.path(export_dir, "genes.csv"), 
          row.names = FALSE, 
          quote = FALSE)

# 3. Save cell barcodes
write.csv(data.frame(barcode = colnames(counts)), 
          file.path(export_dir, "barcodes.csv"), 
          row.names = FALSE, 
          quote = FALSE)

# 4. Save metadata
write.csv(data@meta.data, 
          file.path(export_dir, "metadata.csv"), 
          row.names = TRUE,
          quote = FALSE)

# 5. Optional: Check what other assays/reductions exist
print(names(data@assays))
print(names(data@reductions))

# 6. Optional: Export dimensionality reductions if they exist
if ("pca" %in% names(data@reductions)) {
    write.csv(Embeddings(data, reduction = "pca"),
              file.path(export_dir, "pca.csv"),
              row.names = TRUE)
}

if ("umap" %in% names(data@reductions)) {
    write.csv(Embeddings(data, reduction = "umap"),
              file.path(export_dir, "umap.csv"),
              row.names = TRUE)
}

print("Export complete!")